# CPTAC PDAC Transcriptome Preprocessing

This notebook preprocesses CPTAC PDAC RNA expression data into tumor and normal transcriptome matrices for downstream multi-omics harmonization.

## Inputs

- RNA matrices: `mRNA_cancer.txt`, `mRNA_normal.txt`
- Metadata: `clinical_CPTAC.tsv`
- Mutations for KRAS VAF filtering: `Mutation_results.maf.txt`

## Outputs

- `rna_cancer_pre_harmonized.txt`
- `rna_normal_pre_harmonized.txt`

## Pipeline

1. Load tumor and normal RNA matrices.
2. Remove low-purity tumor samples using the lowest observed KRAS VAF from the CPTAC mutation MAF, with a manual exclusion for C3N-01012 based on low neoplastic cellularity reported for the dataset.
3. Remove genes with more than 30% zero values in either condition.
4. Save pre-harmonized transcriptome matrices for the downstream multi-omics harmonization step.

## Setup

Notebook helper functions are imported from `Pipeline/Data Preprocessing/utils/rna_processing.py`. Imports and configuration are kept together so the notebook can be rerun top-to-bottom with explicit paths and thresholds.

In [1]:
import pandas as pd
from pathlib import Path

from utils.rna_processing import (
    filter_genes_by_zero_fraction,
    filter_samples_by_kras_vaf,
    load_rna_expression_matrices,
    save_processed_rna_matrices,
)

In [ ]:
DATA_DIR = Path("../../data/raw")
OUT_DIR = Path("../../data/processed/transcriptome")
CLINICAL_CPTAC_FILE = Path("../../data/clinical/clinical_CPTAC.tsv")
MUTATION_MAF_FILE = Path("../../data/clinical/Mutation_results.maf.txt")

RNA_CANCER_FILE = DATA_DIR / "mRNA_cancer.txt"
RNA_NORMAL_FILE = DATA_DIR / "mRNA_normal.txt"
OUT_RNA_CANCER = OUT_DIR / "rna_cancer_pre_harmonized.txt"
OUT_RNA_NORMAL = OUT_DIR / "rna_normal_pre_harmonized.txt"
MAX_MISSING_FRACTION = 0.3
KRAS_VAF_THRESHOLD = 0.075
MANUAL_KRAS_VAF_EXCLUSIONS = ["C3N-01012"]

## Step 1: Load Data And Filter Samples

In [3]:
rna_cancer, rna_normal = load_rna_expression_matrices(
    RNA_CANCER_FILE,
    RNA_NORMAL_FILE,
)
clinical_data_cptac = pd.read_csv(CLINICAL_CPTAC_FILE, sep='\t')

print("Filtering samples by KRAS VAF...")
rna_cancer, clinical_data_cptac = filter_samples_by_kras_vaf(
    rna_cancer,
    clinical_data_cptac,
    MUTATION_MAF_FILE,
    vaf_threshold=KRAS_VAF_THRESHOLD,
    extra_removed_samples=MANUAL_KRAS_VAF_EXCLUSIONS,
)

Loading data...
  RNA cancer:        28057 genes,  140 samples
  RNA normal:        28057 genes,   21 samples
Filtering samples by KRAS VAF...
Removed 35 / 140 samples using lowest KRAS VAF < 0.075
Removed 34 samples with lowest KRAS VAF < 0.075
Included manual KRAS VAF exclusions: ['C3N-01012']


## Step 2: Filter Genes By Zero Fraction

In [4]:
print(f"\nFiltering RNA genes with >{MAX_MISSING_FRACTION * 100:.0f}% zero values...")
rna_cancer_filt, rna_normal_filt, expressed_mask = filter_genes_by_zero_fraction(
    rna_cancer,
    rna_normal,
    max_zero_fraction=MAX_MISSING_FRACTION,
)


Filtering RNA genes with >30% zero values...
Genes before zero filtering: 28057
Genes removed by zero filtering: 6610
Genes retained after zero filtering: 21447


## Step 3: Save Outputs

In [5]:
save_processed_rna_matrices(
    rna_cancer_filt,
    rna_normal_filt,
    tumor_out=OUT_RNA_CANCER,
    normal_out=OUT_RNA_NORMAL,
)

Saved: ..\..\..\data\processed\rna_cancer_pre_harmonized.txt
Saved: ..\..\..\data\processed\rna_normal_pre_harmonized.txt
